In [ ]:
# import shutil
# from google.colab import drive
# drive.mount('/content/drive')

# shutil.unpack_archive('/content/drive/MyDrive/cnn_dataset.zip', '/content')
# print("cnn_dataset 압축 해제 완료")


In [7]:
# =====================================================================
# 3단계: CNN 분류기 학습 (MobileNetV2 전이학습, freeze / 파인튜닝 두 버전)
# =====================================================================
# 2_crop_bboxes_for_cnn.py 로 만든 cnn_dataset/ 폴더를 사용합니다.
# cnn_dataset/train/plastic, cnn_dataset/train/glass, cnn_dataset/train/can ...
#
# freeze 버전과 파인튜닝 버전을 각각 학습해서 검증 정확도를 비교한 뒤,
# 더 나은 쪽을 최종 모델로 선택하면 됩니다. (Google Colab GPU 권장)
# =====================================================================

import tensorflow as tf
from tensorflow.keras import layers, models

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 15
DATA_DIR = "cnn_dataset"   # 2번 스크립트의 OUTPUT_DIR과 동일 경로로 수정

# --- 데이터 불러오기 ---
train_ds = tf.keras.utils.image_dataset_from_directory(
    f"{DATA_DIR}/train", image_size=IMG_SIZE, batch_size=BATCH_SIZE
)
val_ds = tf.keras.utils.image_dataset_from_directory(
    f"{DATA_DIR}/valid", image_size=IMG_SIZE, batch_size=BATCH_SIZE
)

class_names = train_ds.class_names
print("클래스:", class_names)  # 예: ['can', 'glass', 'plastic']

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

# 데이터 증강: 위치가 살짝씩 달라지는 실제 환경 대비
augment = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
    layers.RandomBrightness(0.15),  # 조명 변화 대비 (유리·캔 반사 이슈 보완)
])
normalization = layers.Rescaling(1.0 / 127.5, offset=-1)  # MobileNetV2 표준 전처리


def build_model(base_trainable=False, fine_tune_at=100):
    base = tf.keras.applications.MobileNetV2(
        input_shape=IMG_SIZE + (3,), include_top=False, weights="imagenet"
    )
    base.trainable = base_trainable
    if base_trainable:
        # fine_tune_at 이전 레이어는 계속 고정, 이후 레이어만 재학습
        for layer in base.layers[:fine_tune_at]:
            layer.trainable = False

    inputs = tf.keras.Input(shape=IMG_SIZE + (3,))
    x = augment(inputs)
    x = normalization(x)
    x = base(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.2)(x)
    outputs = layers.Dense(len(class_names), activation="softmax")(x)

    model = models.Model(inputs, outputs)
    lr = 1e-5 if base_trainable else 1e-3  # 파인튜닝은 학습률을 훨씬 낮게
    model.compile(
        optimizer=tf.keras.optimizers.Adam(lr),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model


# --- 1) freeze 버전: 특징 추출부 고정, 마지막 분류층만 학습 ---
print("\n=== freeze 버전 학습 ===")
freeze_model = build_model(base_trainable=False)
freeze_history = freeze_model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS)
freeze_model.save("model_freeze.keras")

# --- 2) 파인튜닝 버전: 상위 레이어까지 낮은 학습률로 재학습 ---
print("\n=== 파인튜닝 버전 학습 ===")
finetune_model = build_model(base_trainable=True, fine_tune_at=100)
finetune_history = finetune_model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS)
finetune_model.save("model_finetune.keras")

# --- 3) 두 모델 검증 정확도 비교 ---
freeze_loss, freeze_acc = freeze_model.evaluate(val_ds)
finetune_loss, finetune_acc = finetune_model.evaluate(val_ds)
print(f"\nfreeze 검증 정확도:   {freeze_acc:.4f}")
print(f"finetune 검증 정확도: {finetune_acc:.4f}")

best_model = finetune_model if finetune_acc >= freeze_acc else freeze_model
best_name = "finetune" if finetune_acc >= freeze_acc else "freeze"
print(f"\n-> 최종 선택 모델: {best_name} 버전")

# --- 4) 최종 모델을 .tflite로 변환 (Jetson Nano 등 경량 기기 배포용) ---
converter = tf.lite.TFLiteConverter.from_keras_model(best_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]  # 크기/속도 최적화
tflite_model = converter.convert()

with open("classifier.tflite", "wb") as f:
    f.write(tflite_model)

# 클래스 순서도 같이 저장해둬야 나중에 추론 결과 인덱스를 해석할 수 있음
with open("class_names.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(class_names))

print("\n완료: classifier.tflite, class_names.txt 저장됨")

# =====================================================================
# 다음 단계
# 1) best.pt (YOLO), classifier.tflite (CNN), class_names.txt 를
#    Jetson Nano로 옮김
# 2) 실제 카메라로 YOLO 검출 -> 바운딩박스 크롭 -> CNN 추론 파이프라인 테스트
# 3) 정지-촬영 방식이면: IR센서 감지 -> 벨트 정지 -> 촬영 -> 이 파이프라인
#    -> 서보 매핑 -> 벨트 재가동 순서로 통합
# =====================================================================


Found 402 files belonging to 3 classes.
Found 116 files belonging to 3 classes.
클래스: ['metal', 'paper', 'plastic']

=== freeze 버전 학습 ===
9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Epoch 1/15
13/13 ━━━━━━━━━━━━━━━━━━━━ 27s 1s/step - accuracy: 0.4303 - loss: 1.0978 - val_accuracy: 0.4741 - val_loss: 0.9455
Epoch 2/15
13/13 ━━━━━━━━━━━━━━━━━━━━ 1s 62ms/step - accuracy: 0.6766 - loss: 0.7124 - val_accuracy: 0.8190 - val_loss: 0.5510
Epoch 3/15
13/13 ━━━━━━━━━━━━━━━━━━━━ 1s 76ms/step - accuracy: 0.8134 - loss: 0.4735 - val_accuracy: 0.8621 - val_loss: 0.4320
Epoch 4/15
13/13 ━━━━━━━━━━━━━━━━━━━━ 1s 81ms/step - accuracy: 0.8582 - loss: 0.3908 - val_accuracy: 0.8793 - val_loss: 0.3894
Epoch 5/15
13/13 ━━━━━━━━━━━━━━━━━━━━ 1s 66ms/step - accuracy: 0.8458 - loss: 0.3504 - val_accuracy: 0.9052 - val_loss: 0.3399
Epoch 6/15
13/13 ━━━━━━━━━━━━━━━━━━━━ 1s 63ms/step - accuracy: 0.9055 - loss: 0.2836 - val_accuracy: 0.9052 - val_loss: 0.3217
Epoch 7/15
13/13 ━━━━━━━━━━━━━━━━━━━━ 1s 62ms/step - 

In [ ]:
# --- 5) classifier.tflite / class_names.txt 를 Google Drive에 저장 ---
# (안 하면 세션 끊길 때 /content 에 있던 이 파일들이 전부 사라집니다)
from google.colab import drive
drive.mount('/content/drive')

import shutil, os

SAVE_DIR = "/content/drive/MyDrive/waste_cnn_results"
os.makedirs(SAVE_DIR, exist_ok=True)
shutil.copy("classifier.tflite", f"{SAVE_DIR}/classifier.tflite")
shutil.copy("class_names.txt", f"{SAVE_DIR}/class_names.txt")

# 로컬(맥북)에서 자동으로 받아가기 위한 공유 설정 (waste_yolo_results 때와 동일 방식)
from google.colab import auth
auth.authenticate_user()

from googleapiclient.discovery import build
drive_service = build('drive', 'v3')

result = drive_service.files().list(
    q="name='waste_cnn_results' and mimeType='application/vnd.google-apps.folder'",
    spaces='drive', fields='files(id, name)'
).execute()
folder_id = result['files'][0]['id']

drive_service.permissions().create(
    fileId=folder_id,
    body={'type': 'anyone', 'role': 'reader'},
).execute()

print("저장 완료:", SAVE_DIR)
print("폴더 ID (pull_results.py 같은 스크립트로 받아갈 때 사용):")
print(folder_id)

In [37]:
# =====================================================================
# 3단계: CNN 분류기 학습 (MobileNetV2 전이학습, freeze / 파인튜닝 두 버전)
# =====================================================================
# 2_crop_bboxes_for_cnn.py 로 만든 cnn_dataset/ 폴더를 사용합니다.
# cnn_dataset/train/plastic, cnn_dataset/train/glass, cnn_dataset/train/can ...
#
# freeze 버전과 파인튜닝 버전을 각각 학습해서 검증 정확도를 비교한 뒤,
# 더 나은 쪽을 최종 모델로 선택하면 됩니다. (Google Colab GPU 권장)
# =====================================================================

import tensorflow as tf
from tensorflow.keras import layers, models

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 15
DATA_DIR = "cnn_dataset"   # 2번 스크립트의 OUTPUT_DIR과 동일 경로로 수정

# --- 데이터 불러오기 ---
train_ds = tf.keras.utils.image_dataset_from_directory(
    f"{DATA_DIR}/train", image_size=IMG_SIZE, batch_size=BATCH_SIZE
)
val_ds = tf.keras.utils.image_dataset_from_directory(
    f"{DATA_DIR}/valid", image_size=IMG_SIZE, batch_size=BATCH_SIZE
)

class_names = train_ds.class_names
print("클래스:", class_names)  # 예: ['can', 'glass', 'plastic']

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

# 데이터 증강: 위치가 살짝씩 달라지는 실제 환경 대비
augment = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
    layers.RandomBrightness(0.15),  # 조명 변화 대비 (유리·캔 반사 이슈 보완)
])
normalization = layers.Rescaling(1.0 / 127.5, offset=-1)  # MobileNetV2 표준 전처리


def build_model(base_trainable=False, fine_tune_at=100):
    base = tf.keras.applications.MobileNetV2(
        input_shape=IMG_SIZE + (3,), include_top=False, weights="imagenet"
    )
    base.trainable = base_trainable
    if base_trainable:
        # fine_tune_at 이전 레이어는 계속 고정, 이후 레이어만 재학습
        for layer in base.layers[:fine_tune_at]:
            layer.trainable = False

    inputs = tf.keras.Input(shape=IMG_SIZE + (3,))
    x = augment(inputs)
    x = normalization(x)
    x = base(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.2)(x)
    outputs = layers.Dense(len(class_names), activation="softmax")(x)

    model = models.Model(inputs, outputs)
    lr = 1e-5 if base_trainable else 1e-3  # 파인튜닝은 학습률을 훨씬 낮게
    model.compile(
        optimizer=tf.keras.optimizers.Adam(lr),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model


# --- 1) freeze 버전: 특징 추출부 고정, 마지막 분류층만 학습 ---
print("\n=== freeze 버전 학습 ===")
freeze_model = build_model(base_trainable=False)
freeze_history = freeze_model.fit(
    train_ds, validation_data=val_ds, epochs=EPOCHS, verbose=2
)
freeze_model.save("model_freeze.keras")

# --- 2) 파인튜닝 버전: 상위 레이어까지 낮은 학습률로 재학습 ---
print("\n=== 파인튜닝 버전 학습 ===")
finetune_model = build_model(base_trainable=True, fine_tune_at=100)
finetune_history = finetune_model.fit(
    train_ds, validation_data=val_ds, epochs=EPOCHS, verbose=2
)
finetune_model.save("model_finetune.keras")

# --- 3) 두 모델 검증 정확도 비교 ---
freeze_loss, freeze_acc = freeze_model.evaluate(val_ds, verbose=0)
finetune_loss, finetune_acc = finetune_model.evaluate(val_ds, verbose=0)

best_model = finetune_model if finetune_acc >= freeze_acc else freeze_model
best_name = "finetune" if finetune_acc >= freeze_acc else "freeze"

print("\n" + "=" * 50)
print("최종 결과 요약")
print("=" * 50)
print(f"클래스: {class_names}")
print(f"freeze   검증 정확도: {freeze_acc * 100:6.2f}%   (loss {freeze_loss:.4f})")
print(f"finetune 검증 정확도: {finetune_acc * 100:6.2f}%   (loss {finetune_loss:.4f})")
print(f"-> 최종 선택 모델: {best_name} 버전")
print("=" * 50)

# --- 4) 최종 모델을 .tflite로 변환 (Jetson Nano 등 경량 기기 배포용) ---
converter = tf.lite.TFLiteConverter.from_keras_model(best_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]  # 크기/속도 최적화
tflite_model = converter.convert()

with open("classifier.tflite", "wb") as f:
    f.write(tflite_model)

# 클래스 순서도 같이 저장해둬야 나중에 추론 결과 인덱스를 해석할 수 있음
with open("class_names.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(class_names))

print("\n완료: classifier.tflite, class_names.txt 저장됨")

# =====================================================================
# 다음 단계
# 1) best.pt (YOLO), classifier.tflite (CNN), class_names.txt 를
#    Jetson Nano로 옮김
# 2) 실제 카메라로 YOLO 검출 -> 바운딩박스 크롭 -> CNN 추론 파이프라인 테스트
# 3) 정지-촬영 방식이면: IR센서 감지 -> 벨트 정지 -> 촬영 -> 이 파이프라인
#    -> 서보 매핑 -> 벨트 재가동 순서로 통합
# =====================================

Found 500 files belonging to 4 classes.
Found 143 files belonging to 4 classes.
클래스: ['glass', 'metal', 'paper', 'plastic']

=== freeze 버전 학습 ===
Epoch 1/15
16/16 - 26s - 2s/step - accuracy: 0.3140 - loss: 1.5401 - val_accuracy: 0.5455 - val_loss: 1.0558
Epoch 2/15
16/16 - 1s - 75ms/step - accuracy: 0.6080 - loss: 0.9124 - val_accuracy: 0.7483 - val_loss: 0.7258
Epoch 3/15
16/16 - 1s - 70ms/step - accuracy: 0.7280 - loss: 0.6914 - val_accuracy: 0.8042 - val_loss: 0.6048
Epoch 4/15
16/16 - 1s - 66ms/step - accuracy: 0.7940 - loss: 0.5772 - val_accuracy: 0.8182 - val_loss: 0.5469
Epoch 5/15
16/16 - 1s - 66ms/step - accuracy: 0.7920 - loss: 0.5483 - val_accuracy: 0.8182 - val_loss: 0.5164
Epoch 6/15
16/16 - 1s - 66ms/step - accuracy: 0.8020 - loss: 0.5151 - val_accuracy: 0.8462 - val_loss: 0.5080
Epoch 7/15
16/16 - 1s - 67ms/step - accuracy: 0.8420 - loss: 0.4584 - val_accuracy: 0.8182 - val_loss: 0.4844
Epoch 8/15
16/16 - 1s - 66ms/step - accuracy: 0.8520 - loss: 0.3968 - val_accuracy: 0

In [ ]:
# --- 5) classifier.tflite / class_names.txt 를 Google Drive에 저장 ---
# (안 하면 세션 끊길 때 /content 에 있던 이 파일들이 전부 사라집니다)
from google.colab import drive
drive.mount('/content/drive')

import shutil, os

SAVE_DIR = "/content/drive/MyDrive/waste_cnn_results"
os.makedirs(SAVE_DIR, exist_ok=True)
shutil.copy("classifier.tflite", f"{SAVE_DIR}/classifier.tflite")
shutil.copy("class_names.txt", f"{SAVE_DIR}/class_names.txt")

# 로컬(맥북)에서 자동으로 받아가기 위한 공유 설정 (waste_yolo_results 때와 동일 방식)
from google.colab import auth
auth.authenticate_user()

from googleapiclient.discovery import build
drive_service = build('drive', 'v3')

result = drive_service.files().list(
    q="name='waste_cnn_results' and mimeType='application/vnd.google-apps.folder'",
    spaces='drive', fields='files(id, name)'
).execute()
folder_id = result['files'][0]['id']

drive_service.permissions().create(
    fileId=folder_id,
    body={'type': 'anyone', 'role': 'reader'},
).execute()

print("저장 완료:", SAVE_DIR)
print("폴더 ID (pull_results.py 같은 스크립트로 받아갈 때 사용):")
print(folder_id)